In [2]:
! pip install paddlepaddle PyMuPDF


[notice] A new release of pip is available: 23.2.1 -> 25.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [1]:
! pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cpu


Looking in indexes: https://download.pytorch.org/whl/cpu
  Obtaining dependency information for torch from https://download.pytorch.org/whl/cpu/torch-2.6.0%2Bcpu-cp311-cp311-win_amd64.whl.metadata
  Obtaining dependency information for torchvision from https://download.pytorch.org/whl/cpu/torchvision-0.21.0%2Bcpu-cp311-cp311-win_amd64.whl.metadata
  Obtaining dependency information for torchaudio from https://download.pytorch.org/whl/cpu/torchaudio-2.6.0%2Bcpu-cp311-cp311-win_amd64.whl.metadata
  Using cached https://download.pytorch.org/whl/cpu/torchaudio-2.6.0%2Bcpu-cp311-cp311-win_amd64.whl.metadata (6.7 kB)
   ---------------------------------------- 0.0/206.5 MB ? eta -:--:--
   ---------------------------------------- 0.7/206.5 MB 15.3 MB/s eta 0:00:14
   ---------------------------------------- 1.4/206.5 MB 14.5 MB/s eta 0:00:15
   ---------------------------------------- 1.6/206.5 MB 14.2 MB/s eta 0:00:15
   ---------------------------------------- 1.8/206.5 MB 9.7 MB/s eta 0:0


[notice] A new release of pip is available: 23.2.1 -> 25.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [4]:
! pip install paddlepaddle -f https://www.paddlepaddle.org.cn/whl/simple

Looking in links: https://www.paddlepaddle.org.cn/whl/simple



[notice] A new release of pip is available: 23.2.1 -> 25.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import fitz  # PyMuPDF
import numpy as np
import cv2
from paddleocr import PaddleOCR

# Initialize OCR
ocr = PaddleOCR(lang='en', show_log=False)

def pdf_to_images(pdf_path):
    """Convert PDF to images using PyMuPDF (no Poppler needed)"""
    doc = fitz.open(pdf_path)
    images = []
    for page_num in range(len(doc)):
        page = doc.load_page(page_num)
        pix = page.get_pixmap(dpi=300)  # Set DPI here
        img = np.frombuffer(pix.samples, dtype=np.uint8).reshape(pix.h, pix.w, -1)
        images.append(img)
    return images

def preprocess_image(image):
    """Fix contrast and rotation for better OCR"""
    # Convert to grayscale
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    # Thresholding
    thresh = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)[1]
    return thresh

# Convert PDF to images
pdf_path = "C:/Users/hp/Downloads/pdftomp3-main/pdftomp3-main/media/pdfs/play_strore.pdf"
images = pdf_to_images(pdf_path)

# Process each page
full_text = []
for img in images:
    # Preprocess
    processed = preprocess_image(img)
    # Run OCR
    result = ocr.ocr(processed, cls=False)
    # Extract text
    texts = [line[1][0] for line in result[0]]
    full_text.extend(texts)

# Print cleaned text
print("\n".join(full_text))

OSError: [WinError 127] The specified procedure could not be found. Error loading "c:\Users\hp\AppData\Local\Programs\Python\Python311\Lib\site-packages\torch\lib\shm.dll" or one of its dependencies.

In [1]:
! python --version

Python 3.11.7


In [6]:
! pytesseract.pytesseract.tesseract_cmd = r"C:/Program Files/Tesseract-OCR/tesseract.exe"

'pytesseract.pytesseract.tesseract_cmd' is not recognized as an internal or external command,
operable program or batch file.


In [3]:
import fitz  # PyMuPDF
from PIL import Image
import pytesseract
import os

def pdf_to_images_pymupdf(pdf_path, output_folder="images", zoom=2):
    if not os.path.exists(output_folder):
        os.makedirs(output_folder)
        
    doc = fitz.open(pdf_path)
    image_paths = []

    for page_num in range(len(doc)):
        page = doc.load_page(page_num)
        # Render page to a pixmap with scaling (zoom factor)
        pix = page.get_pixmap(matrix=fitz.Matrix(zoom, zoom))
        image_path = os.path.join(output_folder, f"page_{page_num+1}.png")
        pix.save(image_path)
        image_paths.append(image_path)

    return image_paths

def ocr_images(image_paths):
    extracted_text = ""
    for image_path in image_paths:
        text = pytesseract.image_to_string(Image.open(image_path))
        extracted_text += f"\n--- Text from {image_path} ---\n{text}\n"
    return extracted_text

def pdf_to_text(pdf_path, output_text_file="extracted_text.txt"):
    image_paths = pdf_to_images_pymupdf(pdf_path)
    text = ocr_images(image_paths)
    with open(output_text_file, 'w', encoding='utf-8') as f:
        f.write(text)
    print(f"Text extracted and saved to {output_text_file}")

# Example usage
pdf_path = "C:/Users/hp/Desktop/hint paper.pdf"
pdf_to_text(pdf_path)


Text extracted and saved to extracted_text.txt


In [1]:
import fitz  # PyMuPDF
from PIL import Image, ImageEnhance
import pytesseract
import os
import cv2
import numpy as np

# Set Tesseract path if not in system PATH
# pytesseract.pytesseract.tesseract_cmd = r'C:\Program Files\Tesseract-OCR\tesseract.exe'

def preprocess_image(image_path):
    """Enhance image quality for better OCR results"""
    img = cv2.imread(image_path)
    
    # Convert to grayscale
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    
    # Apply adaptive thresholding
    thresh = cv2.adaptiveThreshold(
        gray, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
        cv2.THRESH_BINARY, 11, 2
    )
    
    # Remove noise
    denoised = cv2.fastNlMeansDenoising(thresh, h=10)
    
    # Enhance contrast
    pil_img = Image.fromarray(denoised)
    enhancer = ImageEnhance.Contrast(pil_img)
    enhanced = enhancer.enhance(2.0)
    
    return np.array(enhanced)

def pdf_to_images(pdf_path, output_folder="images", zoom=3):
    if not os.path.exists(output_folder):
        os.makedirs(output_folder)
        
    doc = fitz.open(pdf_path)
    image_paths = []
    
    for page_num in range(len(doc)):
        page = doc.load_page(page_num)
        pix = page.get_pixmap(matrix=fitz.Matrix(zoom, zoom))
        image_path = os.path.join(output_folder, f"page_{page_num+1}.png")
        pix.save(image_path)
        image_paths.append(image_path)
        
    return image_paths

def ocr_images(image_paths, lang='eng'):
    """Perform OCR with preprocessing and LSTM engine"""
    extracted_text = ""
    
    for image_path in image_paths:
        # Preprocess image
        processed_img = preprocess_image(image_path)
        
        # Use Tesseract LSTM engine with config
        text = pytesseract.image_to_string(
            processed_img,
            lang=lang,
            config='--oem 1 --psm 3 -c tessedit_char_whitelist=ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz0123456789.,!? '
        )
        
        extracted_text += f"\n--- Text from {image_path} ---\n{text}\n"
        
    return extracted_text

def extract_text_from_pdf(pdf_path, output_file="extracted1_text.txt"):
    # Convert PDF to images
    image_paths = pdf_to_images(pdf_path, zoom=3)
    
    # Perform OCR with preprocessing
    text = ocr_images(image_paths)
    
    # Save results
    with open(output_file, 'w', encoding='utf-8') as f:
        f.write(text)
        
    print(f"Text extracted and saved to {output_file}")

if __name__ == "__main__":
    # Process both PDFs
    extract_text_from_pdf("C:/Users/hp/Desktop/Agathiyan_192221126_[PDF TO MP3].pdf")
    extract_text_from_pdf("C:/Users/hp/Desktop/hint paper.pdf")

Text extracted and saved to extracted1_text.txt
Text extracted and saved to extracted1_text.txt


In [6]:
import fitz  # PyMuPDF
from PIL import Image, ImageEnhance
import pytesseract
import os
import cv2
import numpy as np

# Set Tesseract path if not in system PATH
# pytesseract.pytesseract.tesseract_cmd = r'C:\Program Files\Tesseract-OCR\tesseract.exe'

def preprocess_image(image_path):
    """Enhance image quality for better OCR results"""
    img = cv2.imread(image_path)
    
    # Convert to grayscale
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    
    # Apply adaptive thresholding
    thresh = cv2.adaptiveThreshold(
        gray, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
        cv2.THRESH_BINARY, 11, 2
    )
    
    # Remove noise
    denoised = cv2.fastNlMeansDenoising(thresh, h=10)
    
    # Enhance contrast
    pil_img = Image.fromarray(denoised)
    enhancer = ImageEnhance.Contrast(pil_img)
    enhanced = enhancer.enhance(2.0)
    
    return np.array(enhanced)

def pdf_to_images(pdf_path, output_folder="images", zoom=4):
    """Convert PDF pages to high-resolution images"""
    if not os.path.exists(output_folder):
        os.makedirs(output_folder)
        
    doc = fitz.open(pdf_path)
    image_paths = []
    
    for page_num in range(len(doc)):
        page = doc.load_page(page_num)
        pix = page.get_pixmap(matrix=fitz.Matrix(zoom, zoom))  # Increased zoom for better resolution
        image_path = os.path.join(output_folder, f"page_{page_num+1}.png")
        pix.save(image_path)
        image_paths.append(image_path)
        
    return image_paths

def ocr_images(image_paths, lang='eng'):
    """Perform OCR with preprocessing and LSTM engine"""
    extracted_text = ""
    
    for image_path in image_paths:
        # Preprocess image
        processed_img = preprocess_image(image_path)
        
        # Debug: Save preprocessed image for inspection
        debug_image_path = f"debug_{os.path.basename(image_path)}"
        cv2.imwrite(debug_image_path, processed_img)
        
        # Use Tesseract LSTM engine with config
        text = pytesseract.image_to_string(
            processed_img,
            lang=lang,
            config='--oem 1 --psm 6'  # Adjusted psm for block text
        )
        
        extracted_text += f"\n--- Text from {image_path} ---\n{text}\n"
        
    return extracted_text

def extract_text_from_pdf(pdf_path, output_file="extracted_text.txt"):
    """Extract text from a PDF and save it to a file"""
    # Convert PDF to images
    image_paths = pdf_to_images(pdf_path, zoom=4)  # Increased zoom for better resolution
    
    # Perform OCR with preprocessing
    text = ocr_images(image_paths)
    
    # Clean and save results
    text = text.encode('utf-8', errors='ignore').decode('utf-8')  # Handle encoding issues
    with open(output_file, 'w', encoding='utf-8') as f:
        f.write(text)
        
    print(f"Text extracted and saved to {output_file}")

if __name__ == "__main__":
    # Process both PDFs
    extract_text_from_pdf("C:/Users/hp/Desktop/Agathiyan_192221126_[PDF TO MP3].pdf")

Text extracted and saved to extracted_text.txt


In [8]:
import fitz  # PyMuPDF
from PIL import Image, ImageEnhance
import pytesseract
import os
import cv2
import numpy as np

# Set Tesseract path if not in system PATH
# pytesseract.pytesseract.tesseract_cmd = r'C:\Program Files\Tesseract-OCR\tesseract.exe'

def preprocess_handwritten_image(image_path):
    """Preprocess images optimized for handwritten text"""
    img = cv2.imread(image_path)
    
    # Convert to grayscale
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    
    # Apply Gaussian blur to reduce noise
    blurred = cv2.GaussianBlur(gray, (5, 5), 0)
    
    # Adaptive thresholding for better contrast
    thresh = cv2.adaptiveThreshold(
        blurred, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
        cv2.THRESH_BINARY_INV, 11, 2  # Inverted for better handwritten visibility
    )
    
    # Morphological operations to close gaps in text
    kernel = np.ones((3, 3), np.uint8)
    morphed = cv2.morphologyEx(thresh, cv2.MORPH_CLOSE, kernel)
    
    # Remove noise with additional denoising
    denoised = cv2.fastNlMeansDenoising(morphed, h=15)
    
    # Enhance contrast (less aggressive for handwritten text)
    pil_img = Image.fromarray(denoised)
    enhancer = ImageEnhance.Contrast(pil_img)
    enhanced = enhancer.enhance(1.5)  # Reduced contrast enhancement
    
    return np.array(enhanced)

def pdf_to_images(pdf_path, output_folder="images", zoom=4):
    """Convert PDF pages to high-resolution images"""
    if not os.path.exists(output_folder):
        os.makedirs(output_folder)
        
    doc = fitz.open(pdf_path)
    image_paths = []
    
    for page_num in range(len(doc)):
        page = doc.load_page(page_num)
        pix = page.get_pixmap(matrix=fitz.Matrix(zoom, zoom))  # High-res for clarity
        image_path = os.path.join(output_folder, f"page_{page_num+1}.png")
        pix.save(image_path)
        image_paths.append(image_path)
        
    return image_paths

def ocr_handwritten_images(image_paths, lang='eng'):
    """Perform OCR optimized for handwritten text"""
    extracted_text = ""
    
    for image_path in image_paths:
        # Preprocess image for handwritten text
        processed_img = preprocess_handwritten_image(image_path)
        
        # Debug: Save preprocessed image for inspection
        debug_image_path = f"debug_{os.path.basename(image_path)}"
        cv2.imwrite(debug_image_path, processed_img)
        
        # Use Tesseract LSTM engine with handwritten-friendly config
        text = pytesseract.image_to_string(
            processed_img,
            lang=lang,
            config='--oem 1 --psm 11 -c tessedit_char_blacklist=‘’“”'  # PSM 11 for sparse text
        )
        
        extracted_text += f"\n--- Text from {image_path} ---\n{text}\n"
        
    return extracted_text

def extract_text_from_pdf(pdf_path, output_file="extracted_handwritten_text.txt"):
    """Extract handwritten text from a PDF and save it to a file"""
    # Convert PDF to images
    image_paths = pdf_to_images(pdf_path, zoom=4)
    
    # Perform OCR with handwritten optimizations
    text = ocr_handwritten_images(image_paths)
    
    # Clean and save results
    text = text.encode('utf-8', errors='ignore').decode('utf-8')
    with open(output_file, 'w', encoding='utf-8') as f:
        f.write(text)
        
    print(f"Handwritten text extracted and saved to {output_file}")

if __name__ == "__main__":
    # Process both PDFs
    # extract_text_from_pdf("C:/Users/hp/Desktop/Agathiyan_192221126_[PDF TO MP3].pdf")
    extract_text_from_pdf("C:/Users/hp/Desktop/hint paper.pdf")

Handwritten text extracted and saved to extracted_handwritten_text.txt


In [4]:
from PIL import Image
import io

In [ ]:
from PIL import Image
import pytesseract
import fitz  # PyMuPDF
import io

# List of supported languages for multi-language extraction
# Add or remove as per your needs — these are good defaults
MULTI_LANGUAGES = "eng+tam+hin+tel+kan+mal+mar+guj+ben+pun+urd+fra+spa+deu+ita+rus+jpn+chi_sim+kor"

def extract_text_from_pdf_multilang(pdf_path, zoom=2):
    doc = fitz.open(pdf_path)
    full_text = ""

    for page_num in range(len(doc)):
        page = doc.load_page(page_num)
        pix = page.get_pixmap(matrix=fitz.Matrix(zoom, zoom))
        img_data = pix.tobytes("png")
        img = Image.open(io.BytesIO(img_data))

        # Perform multi-language OCR
        text = pytesseract.image_to_string(img, lang="tam")
        full_text += f"\n{text.strip()}\n"

    return full_text.strip()


In [18]:
lang=extract_text_and_detect_language("C:/Users/hp/Downloads/pdftomp3-main/pdftomp3-main/media/pdfs/tamilpdf_m3LVYTe_0sI7EoG.pdf")

Extracted text from first page (no OCR): அஆஆஇஈஈஉஊஊஎஏஐஒஓஔஃ
ககஙஙசஞடடணததநனபப
மயரலவவழழளறஷஸஜஹ}
௧௨௩௪௫௬௭௮௯௰௱௲
ககாகாக கீகீ   ெகேகைகெகாேகாெகௗகஃ
நன...
Detected language from native text: ta


In [19]:
print(lang)

ta


In [17]:
TESSERACT_LANG_MAP = {
    "en": "eng",
    "es": "spa",
    "fr": "fra",
    "de": "deu",
    "it": "ita",
    "hi": "hin",
    "zh-cn": "chi_sim",
    "ja": "jpn",
    "ko": "kor",
    "ru": "rus",
    "ta": "tam",
}
TESSERACT_LANG_MAP.get("ta", "eng")

'tam'

In [14]:
extract_text_from_pdf_multilang("C:/Users/hp/Downloads/pdftomp3-main/pdftomp3-main/media/pdfs/tamilpdf_m3LVYTe_0sI7EoG.pdf")

'அஆஆஇஈரஉஊஊஎஏஜஐஒஓஓஎஃ\n\nககஙங சஞ்டடணததநனபப\nமயரலவவழழளறஷஸ ஜஹ க்ஷ\n௧௨௩௪௬௭௬௭௮௯ம௱௯\nககாகாகிகீகீகுக்கூகெகேகைகொகோகெள கஃ\n\nநன்றி'

In [1]:
! pip install pillow pytesseract easyocr opencv-python


[notice] A new release of pip is available: 23.2.1 -> 25.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip
